# 🏭 Random Forest 파쇄 크기 예측 모델

## 배터리 리사이클링 슈레더 — 파쇄 크기 & 균일도 예측

**모델 개요:**
- **알고리즘:** Random Forest Regressor (앙상블 학습)
- **입력:** 14종 센서 → 12개 엔지니어링 피처 (4센서 그룹 × 3통계)
- **출력:** 파쇄 크기 (mm), 균일도 (%)
- **데이터:** 365일 × 10분 간격 = **52,560 샘플** (프로덕션 규모)

**센서 그룹:**
| 그룹 | 센서 | 설명 |
|------|------|------|
| CUR | CUR_A, CUR_B | 모터 전류 (A축/B축) |
| SPD | SPD_A, SPD_B | 모터 속도 RPM (A축/B축) |
| VIB | VIB_A/B (x,y,z) | 3축 가속도 진동 6채널 → RMS |
| SCL | SCL_weight | 처리량 스케일 (kg) |

**Random Forest 핵심 원리:**
1. **Bagging:** Bootstrap 샘플로 200개 Decision Tree 독립 학습
2. **Feature Randomness:** 각 분할에서 sqrt(12) ≈ 3~4개 피처만 후보로 사용
3. **앙상블 평균:** 모든 트리의 예측값을 평균하여 과적합 방지

## Step 0. 라이브러리 설치 및 임포트

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# matplotlib 영문 설정
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("라이브러리 로드 완료")

## Step 1. 데이터 생성 (365일, 10분 간격 = 52,560 샘플)

슈레더 14종 센서 시뮬레이션 데이터를 생성합니다.
- 가동 패턴: 평일 08~18시 정상가동, 야간 저부하, 주말 정지
- 칼날 마모: 0% → 30% 비선형 증가 (365일)
- 이상 이벤트: 베어링 이상, 발화, 분진 급상승, 가스 누출, 이물질 끼임

In [ ]:
# ──────────────────────────────────────────────
# 상수 정의
# ──────────────────────────────────────────────
_BASE_RPM_A = 1200.0      # A축 정격 RPM
_BASE_RPM_B = 800.0       # B축 정격 RPM (역회전, 감속)
_BASE_CUR_A = 85.0        # A축 정격 전류 (A)
_BASE_CUR_B = 60.0        # B축 정격 전류 (A)
_BASE_VIB = 2.5           # 정상 진동 RMS (mm/s)
_BASE_TEMP = 35.0         # 정상 베어링 온도 (°C)
_BASE_IR_TEMP = 45.0      # 정상 IR 표면 온도 (°C)
_BASE_DUST = 5.0          # 정상 분진 농도 (mg/m³)
_BASE_GAS_VOC = 10.0      # 정상 VOC (ppm)
_BASE_GAS_H2 = 2.0        # 정상 H2 (ppm)
_BASE_GAS_CO = 3.0        # 정상 CO (ppm)
_BASE_WEIGHT = 150.0      # 10분당 처리량 (kg)

# 이상 이벤트 확률
_P_BEARING_ANOMALY = 0.003
_P_FIRE_EVENT = 0.001
_P_DUST_SPIKE = 0.005
_P_GAS_LEAK = 0.002
_P_JAM_EVENT = 0.004


# ──────────────────────────────────────────────
# 헬퍼 함수
# ──────────────────────────────────────────────
def _operating_mask(timestamps):
    """가동 마스크 생성: 평일 08-18시 정상가동, 야간/주말은 대기 또는 정지."""
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])
    operating = np.ones(len(timestamps), dtype=float)
    operating[dow >= 5] = 0.0                              # 주말 정지
    night_mask = (hours < 8) | (hours >= 18)
    operating[night_mask & (dow < 5)] = 0.15               # 평일 야간
    operating[(hours == 12) & (dow < 5)] = 0.6             # 점심
    return operating


def _daily_cycle(hours, phase_shift=0.0):
    """일간 사인파 패턴 (낮에 높고 밤에 낮음)"""
    return np.sin(2 * np.pi * hours / 24 - np.pi / 2 + phase_shift)


def _wear_trend(n_points, days, max_wear_pct=30.0):
    """칼날 마모 트렌드 (0% → max_wear_pct%, 가속 마모 커브)"""
    t_norm = np.linspace(0, 1, n_points)
    return max_wear_pct * t_norm ** 1.3


def _inject_events(n_points, rng, event_prob, duration_range=(3, 15)):
    """이벤트 마스크 및 강도 생성"""
    mask = np.zeros(n_points, dtype=bool)
    intensity = np.zeros(n_points)
    i = 0
    while i < n_points:
        if rng.random() < event_prob:
            dur = rng.integers(duration_range[0], duration_range[1] + 1)
            end = min(i + dur, n_points)
            mask[i:end] = True
            event_len = end - i
            peak = rng.uniform(0.5, 1.0)
            ramp = np.concatenate([
                np.linspace(0, peak, event_len // 2 + 1),
                np.linspace(peak, 0, event_len - event_len // 2)
            ])[:event_len]
            intensity[i:end] = ramp
            i = end + rng.integers(50, 200)
        else:
            i += 1
    return mask, intensity


# ──────────────────────────────────────────────
# 메인 데이터 생성 함수
# ──────────────────────────────────────────────
def generate_shredder_full_data(days=365, freq_minutes=10, seed=42):
    """
    슈레더 전체 센서 데이터 시뮬레이션.
    Returns: DataFrame (14종 센서 + 칼날 마모율 + 타임스탬프 + 이벤트 라벨)
    """
    rng = np.random.default_rng(seed)
    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(start='2026-01-01', periods=n_points, freq=f'{freq_minutes}min')

    hours = np.array([ts.hour for ts in timestamps])
    t = np.arange(n_points)

    # 가동 마스크
    op = _operating_mask(timestamps)

    # 칼날 마모 트렌드
    wear = _wear_trend(n_points, days, max_wear_pct=30.0)
    wear_factor = 1.0 + wear / 100.0  # 1.0 → 1.3

    # 이상 이벤트
    bearing_mask, bearing_int = _inject_events(n_points, rng, _P_BEARING_ANOMALY, (5, 20))
    fire_mask, fire_int = _inject_events(n_points, rng, _P_FIRE_EVENT, (3, 10))
    dust_mask, dust_int = _inject_events(n_points, rng, _P_DUST_SPIKE, (5, 25))
    gas_mask, gas_int = _inject_events(n_points, rng, _P_GAS_LEAK, (10, 40))
    jam_mask, jam_int = _inject_events(n_points, rng, _P_JAM_EVENT, (2, 8))

    # 일간 사이클
    daily = _daily_cycle(hours)
    daily_shifted = _daily_cycle(hours, phase_shift=np.pi / 6)

    # SPD: 모터 속도 (RPM)
    spd_a = (_BASE_RPM_A * op + 30.0 * daily * op
             + rng.normal(0, 8, n_points) * op - 400.0 * jam_int * op)
    spd_a = np.clip(spd_a, 0, _BASE_RPM_A * 1.15)

    spd_b = (_BASE_RPM_B * op + 20.0 * daily * op
             + rng.normal(0, 6, n_points) * op - 300.0 * jam_int * op)
    spd_b = np.clip(spd_b, 0, _BASE_RPM_B * 1.15)

    # CUR: 모터 전류 (A)
    cur_a = (_BASE_CUR_A * op * wear_factor + 8.0 * daily * op
             + rng.normal(0, 2.0, n_points) * op + 25.0 * jam_int * op)
    cur_a = np.clip(cur_a, 0, 180)

    cur_b = (_BASE_CUR_B * op * wear_factor + 5.0 * daily * op
             + rng.normal(0, 1.5, n_points) * op + 18.0 * jam_int * op)
    cur_b = np.clip(cur_b, 0, 130)

    # VIB: 3축 진동 (mm/s)
    def _make_vib(base, axis_weight, bearing_scale):
        vib = (base * op * wear_factor * axis_weight
               + 0.4 * daily * op * axis_weight
               + rng.normal(0, 0.25, n_points) * op
               + bearing_scale * bearing_int * op
               + 1.5 * jam_int * op)
        return np.clip(vib, 0, 30)

    vib_a_x = _make_vib(_BASE_VIB, 1.0, 12.0)
    vib_a_y = _make_vib(_BASE_VIB, 0.8, 10.0)
    vib_a_z = _make_vib(_BASE_VIB, 0.5, 6.0)
    vib_b_x = _make_vib(_BASE_VIB * 0.8, 1.0, 9.0)
    vib_b_y = _make_vib(_BASE_VIB * 0.8, 0.8, 7.5)
    vib_b_z = _make_vib(_BASE_VIB * 0.8, 0.5, 4.5)

    # TMP: 온도 (°C)
    tmp_ir1 = (_BASE_IR_TEMP * np.maximum(op, 0.4) + 5.0 * daily * op
               + wear * 0.1 * op + rng.normal(0, 1.2, n_points)
               + 80.0 * fire_int + 8.0 * jam_int * op)
    tmp_ir2 = tmp_ir1 + rng.normal(0, 1.5, n_points) - 2.0
    tmp_ir1 = np.clip(tmp_ir1, 15, 350)
    tmp_ir2 = np.clip(tmp_ir2, 15, 340)

    tmp_a = (_BASE_TEMP * np.maximum(op, 0.5) + 3.0 * daily_shifted * op
             + wear * 0.08 * op + rng.normal(0, 0.6, n_points)
             + 15.0 * bearing_int * op + 30.0 * fire_int)
    tmp_a = np.clip(tmp_a, 15, 150)

    tmp_b = (_BASE_TEMP * 0.9 * np.maximum(op, 0.5) + 2.5 * daily_shifted * op
             + wear * 0.06 * op + rng.normal(0, 0.5, n_points)
             + 12.0 * bearing_int * op + 25.0 * fire_int)
    tmp_b = np.clip(tmp_b, 15, 140)

    # DST: 분진 농도 (mg/m³)
    dst_1 = (_BASE_DUST * op * wear_factor + 1.5 * daily * op
             + rng.normal(0, 0.8, n_points) * op
             + rng.exponential(0.5, n_points) * op
             + 40.0 * dust_int * op + 5.0 * jam_int * op)
    dst_1 = np.clip(dst_1, 0, 80)

    # GAS: 가스 센서 (ppm)
    gas_voc = (_BASE_GAS_VOC * np.maximum(op, 0.3) + 2.0 * daily * op
               + rng.normal(0, 1.0, n_points) + 200.0 * gas_int + 15.0 * fire_int)
    gas_voc = np.clip(gas_voc, 0, 500)

    gas_h2 = (_BASE_GAS_H2 * np.maximum(op, 0.2) + 0.3 * daily * op
              + rng.normal(0, 0.3, n_points) + 80.0 * gas_int + 20.0 * fire_int)
    gas_h2 = np.clip(gas_h2, 0, 200)

    gas_co = (_BASE_GAS_CO * np.maximum(op, 0.2) + 0.5 * daily * op
              + rng.normal(0, 0.4, n_points) + 50.0 * gas_int + 40.0 * fire_int)
    gas_co = np.clip(gas_co, 0, 200)

    # SCL: 처리량 (kg/10분)
    scl_weight = (_BASE_WEIGHT * op + 15.0 * daily * op
                  + rng.normal(0, 5.0, n_points) * op
                  - 80.0 * jam_int * op - wear * 0.3 * op)
    scl_weight = np.clip(scl_weight, 0, 250)

    # 이벤트 라벨
    event_labels = np.full(n_points, 'normal', dtype=object)
    event_labels[bearing_mask] = 'bearing_anomaly'
    event_labels[fire_mask] = 'fire_event'
    event_labels[dust_mask] = 'dust_spike'
    event_labels[gas_mask] = 'gas_leak'
    event_labels[jam_mask] = 'jam_event'

    df = pd.DataFrame({
        'timestamp': timestamps,
        'VIB_A_x': np.round(vib_a_x, 3), 'VIB_A_y': np.round(vib_a_y, 3),
        'VIB_A_z': np.round(vib_a_z, 3), 'VIB_B_x': np.round(vib_b_x, 3),
        'VIB_B_y': np.round(vib_b_y, 3), 'VIB_B_z': np.round(vib_b_z, 3),
        'CUR_A': np.round(cur_a, 2), 'CUR_B': np.round(cur_b, 2),
        'SPD_A': np.round(spd_a, 1), 'SPD_B': np.round(spd_b, 1),
        'TMP_IR1': np.round(tmp_ir1, 1), 'TMP_IR2': np.round(tmp_ir2, 1),
        'TMP_A': np.round(tmp_a, 1), 'TMP_B': np.round(tmp_b, 1),
        'DST_1': np.round(dst_1, 2), 'GAS_VOC': np.round(gas_voc, 1),
        'GAS_H2': np.round(gas_h2, 1), 'GAS_CO': np.round(gas_co, 1),
        'SCL_weight': np.round(scl_weight, 1),
        'blade_wear_pct': np.round(wear, 2),
        'event_label': event_labels,
    })
    return df

print("데이터 생성 함수 정의 완료")

In [ ]:
# 365일 데이터 생성 (52,560 샘플)
df = generate_shredder_full_data(days=365, freq_minutes=10)

print(f"생성된 데이터 형상: {df.shape}")
print(f"기간: {df['timestamp'].iloc[0]} ~ {df['timestamp'].iloc[-1]}")
print(f"샘플 수: {len(df):,}개")
print(f"\n센서 컬럼 ({len(df.columns) - 2}종):")  # timestamp, event_label 제외
print(df.columns.tolist())
print(f"\n이벤트 분포:")
print(df['event_label'].value_counts())
print(f"\n데이터 요약 통계:")
df.describe().round(2)

## Step 2. 원시 데이터 시각화

파쇄 크기 예측에 사용할 주요 센서 (CUR, SPD, VIB, SCL)의 시계열 패턴을 확인합니다.

In [ ]:
# 센서 개요 — 주요 4그룹 시계열
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# CUR (Motor Current)
axes[0].plot(df['timestamp'], df['CUR_A'], alpha=0.6, linewidth=0.5, label='CUR_A')
axes[0].plot(df['timestamp'], df['CUR_B'], alpha=0.6, linewidth=0.5, label='CUR_B')
axes[0].set_ylabel('Current (A)')
axes[0].set_title('Sensor Overview — 365 Days at 10-min Intervals (52,560 samples)', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')

# SPD (Motor Speed)
axes[1].plot(df['timestamp'], df['SPD_A'], alpha=0.6, linewidth=0.5, color='#e67e22', label='SPD_A')
axes[1].plot(df['timestamp'], df['SPD_B'], alpha=0.6, linewidth=0.5, color='#27ae60', label='SPD_B')
axes[1].set_ylabel('Speed (RPM)')
axes[1].legend(loc='upper right')

# VIB (Vibration RMS)
vib_rms = np.sqrt((df['VIB_A_x']**2 + df['VIB_A_y']**2 + df['VIB_A_z']**2 +
                   df['VIB_B_x']**2 + df['VIB_B_y']**2 + df['VIB_B_z']**2) / 6)
axes[2].plot(df['timestamp'], vib_rms, alpha=0.6, linewidth=0.5, color='#e74c3c', label='VIB RMS (6ch)')
axes[2].set_ylabel('Vibration RMS (mm/s)')
axes[2].legend(loc='upper right')

# SCL (Weight)
axes[3].plot(df['timestamp'], df['SCL_weight'], alpha=0.6, linewidth=0.5, color='#8e44ad', label='SCL_weight')
axes[3].set_ylabel('Throughput (kg)')
axes[3].set_xlabel('Timestamp')
axes[3].legend(loc='upper right')

plt.tight_layout()
plt.show()
print("센서 개요 시각화 완료")

In [ ]:
# 운전 패턴 분석 — 일간/주간 패턴 + 마모 트렌드
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 일간 패턴 (시간대별 전류 평균)
hourly_cur = df.groupby(df['timestamp'].dt.hour)['CUR_A'].mean()
axes[0].bar(hourly_cur.index, hourly_cur.values, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Current A (A)')
axes[0].set_title('Daily Operating Pattern — CUR_A', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(0, 24, 2))

# 주간 패턴 (요일별 처리량 평균)
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly_scl = df.groupby(df['timestamp'].dt.dayofweek)['SCL_weight'].mean()
colors_dow = ['#2ecc71'] * 5 + ['#e74c3c'] * 2
axes[1].bar(range(7), weekly_scl.values, color=colors_dow, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Throughput (kg)')
axes[1].set_title('Weekly Operating Pattern — SCL_weight', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_names)

# 마모 트렌드
axes[2].plot(df['timestamp'], df['blade_wear_pct'], color='#e74c3c', linewidth=1.5)
axes[2].set_xlabel('Timestamp')
axes[2].set_ylabel('Blade Wear (%)')
axes[2].set_title('Blade Wear Trend (365 days)', fontsize=12, fontweight='bold')
axes[2].fill_between(df['timestamp'], df['blade_wear_pct'], alpha=0.2, color='#e74c3c')

plt.tight_layout()
plt.show()
print("운전 패턴 분석 완료")

## Step 3. 피처 엔지니어링 (12개 피처)

4개 센서 그룹에서 각 3개 통계량 (mean, std, max)을 추출합니다.

| 피처 | 센서 원본 | 계산 방법 |
|------|----------|----------|
| CUR_mean/std/max | CUR_A, CUR_B | A/B축 평균, 차이, 최대값 |
| SPD_mean/std/max | SPD_A, SPD_B | A/B축 평균, 차이, 최대값 |
| VIB_mean/std/max | VIB 6채널 | 6채널 평균, 표준편차, 최대값 |
| SCL_mean/std/max | SCL_weight | 현재값, 롤링 표준편차, 롤링 최대값 |

In [ ]:
def extract_12_features(df):
    """
    CUR, SPD, VIB, SCL 센서에서 12개 피처 추출.
    각 센서별 mean, std, max → 4그룹 x 3통계 = 12피처
    """
    features = pd.DataFrame(index=df.index)

    # CUR (전류): A축+B축 통계
    features['CUR_mean'] = (df['CUR_A'] + df['CUR_B']) / 2
    features['CUR_std'] = np.abs(df['CUR_A'] - df['CUR_B'])
    features['CUR_max'] = np.maximum(df['CUR_A'], df['CUR_B'])

    # SPD (속도): A축+B축 통계
    features['SPD_mean'] = (df['SPD_A'] + df['SPD_B']) / 2
    features['SPD_std'] = np.abs(df['SPD_A'] - df['SPD_B'])
    features['SPD_max'] = np.maximum(df['SPD_A'], df['SPD_B'])

    # VIB (진동): 6채널 통계
    vib_cols = ['VIB_A_x', 'VIB_A_y', 'VIB_A_z', 'VIB_B_x', 'VIB_B_y', 'VIB_B_z']
    vib_data = df[vib_cols]
    features['VIB_mean'] = vib_data.mean(axis=1)
    features['VIB_std'] = vib_data.std(axis=1)
    features['VIB_max'] = vib_data.max(axis=1)

    # SCL (스케일/무게): 롤링 윈도우 통계 (window=6 = 1시간)
    features['SCL_mean'] = df['SCL_weight']
    features['SCL_std'] = df['SCL_weight'].rolling(6, min_periods=1).std().fillna(0)
    features['SCL_max'] = df['SCL_weight'].rolling(6, min_periods=1).max().fillna(df['SCL_weight'])

    return features


# 12개 피처 추출
features = extract_12_features(df)

# 피처 그룹 정의 (시각화 색상용)
FEATURE_GROUPS = {
    'CUR': ['CUR_mean', 'CUR_std', 'CUR_max'],
    'SPD': ['SPD_mean', 'SPD_std', 'SPD_max'],
    'VIB': ['VIB_mean', 'VIB_std', 'VIB_max'],
    'SCL': ['SCL_mean', 'SCL_std', 'SCL_max'],
}

FEATURE_NAMES_EN = features.columns.tolist()
FEATURE_NAMES_KO = [
    '전류 평균', '전류 표준편차', '전류 최대값',
    '속도 평균', '속도 표준편차', '속도 최대값',
    '진동 평균', '진동 표준편차', '진동 최대값',
    '무게 평균', '무게 표준편차', '무게 최대값',
]

print(f"피처 행렬 형상: {features.shape}")
print(f"피처 수: {features.shape[1]}개\n")

for i, (en, ko) in enumerate(zip(FEATURE_NAMES_EN, FEATURE_NAMES_KO), 1):
    print(f"  {i:2d}. {en:12s} ({ko})")

print(f"\n피처 통계:")
features.describe().round(2)

## Step 4. 타겟 시뮬레이션 (파쇄 크기 & 균일도)

**파쇄 크기 (mm):** RPM, 전류, 진동, 마모의 물리적 관계를 반영
- 전류 높음 → 파쇄 크기 감소 (강한 분쇄력)
- 속도 높음 → 파쇄 크기 감소 (빠른 회전)
- 진동 높음 → 파쇄 크기 불규칙 증가
- 무게 높음 → 파쇄 크기 증가 (과부하)
- 범위: 5~50 mm

**균일도 (%):** 진동과 마모에 역비례
- 진동 낮고 속도 안정 → 균일도 높음
- 범위: 40~99%

In [ ]:
def simulate_shred_targets(features, blade_wear):
    """
    파쇄 크기(mm)와 균일도(%) 시뮬레이션.
    물리적 상관관계를 반영한 현실적 타겟 생성.
    """
    n = len(features)
    rng = np.random.default_rng(123)

    # ── 파쇄 크기 (mm) ──
    # 전류↑ → 크기↓, 속도↑ → 크기↓, 진동↑ → 크기↑(불규칙), 무게↑ → 크기↑
    shred_size = (
        25.0
        - 0.04 * (features['CUR_mean'].values - features['CUR_mean'].median())
        - 0.008 * (features['SPD_mean'].values - features['SPD_mean'].median())
        + 0.8 * (features['VIB_mean'].values - features['VIB_mean'].median())
        + 0.03 * (features['SCL_mean'].values - features['SCL_mean'].median())
        + 0.05 * blade_wear  # 마모 증가 → 파쇄 크기 증가
        + rng.normal(0, 0.8, n)
    )
    shred_size = np.clip(shred_size, 5, 50)

    # ── 균일도 (%) ──
    # 진동 낮고 속도 안정적이면 높음, 마모 증가하면 감소
    uniformity = (
        88.0
        - 1.5 * features['VIB_std'].values
        - 0.3 * features['SPD_std'].values
        - 0.2 * features['CUR_std'].values
        - 0.08 * blade_wear  # 마모 증가 → 균일도 감소
        + rng.normal(0, 1.5, n)
    )
    uniformity = np.clip(uniformity, 40, 99)

    return np.round(shred_size, 2), np.round(uniformity, 2)


# 타겟 생성
shred_size, uniformity = simulate_shred_targets(features, df['blade_wear_pct'].values)

print(f"파쇄 크기 통계:")
print(f"  범위: {shred_size.min():.1f} ~ {shred_size.max():.1f} mm")
print(f"  평균: {np.mean(shred_size):.2f} mm")
print(f"  표준편차: {np.std(shred_size):.2f} mm")

print(f"\n균일도 통계:")
print(f"  범위: {uniformity.min():.1f} ~ {uniformity.max():.1f} %")
print(f"  평균: {np.mean(uniformity):.2f} %")
print(f"  표준편차: {np.std(uniformity):.2f} %")

## Step 5. 학습/테스트 분할 (80/20 시간순)

시계열 데이터이므로 **시간 순서를 유지하여 분할**합니다 (랜덤 셔플 없음).
- 학습 데이터: 처음 80% (약 292일)
- 테스트 데이터: 나머지 20% (약 73일)

In [ ]:
# 시간 순서 유지 분할 (80/20)
X = features.values
y_size = shred_size
y_unif = uniformity
timestamps_arr = df['timestamp'].values

split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_size_train, y_size_test = y_size[:split_idx], y_size[split_idx:]
y_unif_train, y_unif_test = y_unif[:split_idx], y_unif[split_idx:]
ts_train, ts_test = timestamps_arr[:split_idx], timestamps_arr[split_idx:]

print(f"학습/테스트 분할 (시간순, 80/20):")
print(f"  학습 데이터: {len(X_train):,}건 ({pd.Timestamp(ts_train[0]).strftime('%Y-%m-%d')} ~ {pd.Timestamp(ts_train[-1]).strftime('%Y-%m-%d')})")
print(f"  테스트 데이터: {len(X_test):,}건 ({pd.Timestamp(ts_test[0]).strftime('%Y-%m-%d')} ~ {pd.Timestamp(ts_test[-1]).strftime('%Y-%m-%d')})")
print(f"\n  피처 수: {X_train.shape[1]}개")
print(f"  학습 파쇄크기 평균: {y_size_train.mean():.2f} mm")
print(f"  테스트 파쇄크기 평균: {y_size_test.mean():.2f} mm")

## Step 6. Random Forest 모델 학습

두 개의 Random Forest 모델을 학습합니다:
1. **파쇄 크기 예측 모델** — 목표: 파쇄물 크기 (mm)
2. **균일도 예측 모델** — 목표: 파쇄 균일도 (%)

**하이퍼파라미터:**
- `n_estimators=200`: 200개 Decision Tree 앙상블
- `max_depth=12`: 트리 최대 깊이 12 (과적합 방지)
- `max_features='sqrt'`: 각 분할에서 sqrt(12) ≈ 3~4개 피처 사용
- `random_state=42`: 재현성 보장

In [ ]:
import time

# ── 모델 1: 파쇄 크기 예측 ──
print("=" * 60)
print("  모델 1: Random Forest — 파쇄 크기 예측")
print("=" * 60)

rf_size = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

start = time.time()
rf_size.fit(X_train, y_size_train)
t_size = time.time() - start

y_size_pred = rf_size.predict(X_test)
print(f"  학습 시간: {t_size:.2f}초")
print(f"  트리 수: {rf_size.n_estimators}")
print(f"  최대 깊이: {rf_size.max_depth}")
print(f"  피처 수: {rf_size.n_features_in_}")

# ── 모델 2: 균일도 예측 ──
print(f"\n{'=' * 60}")
print("  모델 2: Random Forest — 균일도 예측")
print("=" * 60)

rf_unif = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

start = time.time()
rf_unif.fit(X_train, y_unif_train)
t_unif = time.time() - start

y_unif_pred = rf_unif.predict(X_test)
print(f"  학습 시간: {t_unif:.2f}초")
print(f"  트리 수: {rf_unif.n_estimators}")

print(f"\n총 학습 시간: {t_size + t_unif:.2f}초")

## Step 7. 모델 평가 (MAE, RMSE, R²)

두 모델의 성능을 3가지 메트릭으로 평가합니다:
- **MAE (Mean Absolute Error):** 평균 절대 오차
- **RMSE (Root Mean Squared Error):** 평균 제곱근 오차
- **R² (Coefficient of Determination):** 결정 계수 (1에 가까울수록 좋음)

In [ ]:
# ── 파쇄 크기 모델 평가 ──
mae_size = mean_absolute_error(y_size_test, y_size_pred)
rmse_size = np.sqrt(mean_squared_error(y_size_test, y_size_pred))
r2_size = r2_score(y_size_test, y_size_pred)

# ── 균일도 모델 평가 ──
mae_unif = mean_absolute_error(y_unif_test, y_unif_pred)
rmse_unif = np.sqrt(mean_squared_error(y_unif_test, y_unif_pred))
r2_unif = r2_score(y_unif_test, y_unif_pred)

print("=" * 60)
print("  모델 성능 평가 결과")
print("=" * 60)

print(f"\n  [파쇄 크기 예측 모델]")
print(f"    MAE  = {mae_size:.4f} mm")
print(f"    RMSE = {rmse_size:.4f} mm")
print(f"    R²   = {r2_size:.4f}")

print(f"\n  [균일도 예측 모델]")
print(f"    MAE  = {mae_unif:.4f} %")
print(f"    RMSE = {rmse_unif:.4f} %")
print(f"    R²   = {r2_unif:.4f}")

# 성능 요약 테이블
eval_df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Shred Size (mm)': [f'{mae_size:.4f}', f'{rmse_size:.4f}', f'{r2_size:.4f}'],
    'Uniformity (%)': [f'{mae_unif:.4f}', f'{rmse_unif:.4f}', f'{r2_unif:.4f}'],
})
print(f"\n성능 요약 테이블:")
eval_df

## Step 8. 결과 시각화

각 차트를 개별 셀에서 생성합니다.

### 8-1. 예측 vs 실제 파쇄 크기 시계열

테스트 기간(약 73일)의 실제값과 예측값을 시계열로 비교합니다.

In [ ]:
# 8-1. Predicted vs Actual — Shred Size Time Series
fig, ax = plt.subplots(figsize=(16, 5))

ts_plot = pd.to_datetime(ts_test)
ax.plot(ts_plot, y_size_test, alpha=0.6, linewidth=0.8, color='#2563eb', label='Actual')
ax.plot(ts_plot, y_size_pred, alpha=0.7, linewidth=0.8, color='#dc2626', label='Predicted (RF)')

# 오차 영역 표시
ax.fill_between(ts_plot, y_size_test, y_size_pred, alpha=0.15, color='#f59e0b', label='Error')

ax.set_xlabel('Timestamp')
ax.set_ylabel('Shred Size (mm)')
ax.set_title(f'Shred Size: Actual vs Predicted (Test Period, R²={r2_size:.4f}, MAE={mae_size:.2f}mm)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("8-1. 파쇄 크기 시계열 비교 완료")

### 8-2. 피처 중요도 (12개 피처, 그룹별 색상)

Random Forest가 파쇄 크기 예측에 어떤 센서 피처를 가장 중요하게 사용하는지 확인합니다.

In [ ]:
# 8-2. Feature Importance (12 features, color-coded by sensor group)
importance = pd.Series(rf_size.feature_importances_, index=FEATURE_NAMES_EN)
importance = importance.sort_values(ascending=True)  # ascending for horizontal bar

# 그룹별 색상 매핑
GROUP_COLORS = {
    'CUR': '#e74c3c',  # Red
    'SPD': '#3498db',  # Blue
    'VIB': '#2ecc71',  # Green
    'SCL': '#9b59b6',  # Purple
}

def get_group_color(feature_name):
    for group, feats in FEATURE_GROUPS.items():
        if feature_name in feats:
            return GROUP_COLORS[group]
    return '#95a5a6'

colors = [get_group_color(f) for f in importance.index]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(importance)), importance.values, color=colors, edgecolor='white', height=0.7)

# 피처 라벨 (EN + KO)
ko_map = dict(zip(FEATURE_NAMES_EN, FEATURE_NAMES_KO))
labels = [f'{name}' for name in importance.index]
ax.set_yticks(range(len(importance)))
ax.set_yticklabels(labels, fontsize=10)

# 값 표시
for i, (val, name) in enumerate(zip(importance.values, importance.index)):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Feature Importance — Shred Size Prediction (12 Features)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, title='Sensor Group')

plt.tight_layout()
plt.show()
print("8-2. 피처 중요도 시각화 완료")

### 8-3. 실제 vs 예측 산점도 (R² 라인 포함)

완벽한 예측선(y=x)과 비교하여 모델 정확도를 시각적으로 확인합니다.

In [ ]:
# 8-3. Actual vs Predicted Scatter (with R² line)
fig, ax = plt.subplots(figsize=(8, 8))

errors = np.abs(y_size_test - y_size_pred)
scatter = ax.scatter(y_size_test, y_size_pred, c=errors, cmap='RdYlGn_r',
                     alpha=0.4, s=10, edgecolors='none')

# Perfect prediction line
min_val = min(y_size_test.min(), y_size_pred.min())
max_val = max(y_size_test.max(), y_size_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Perfect Prediction (y=x)')

# Linear fit
z = np.polyfit(y_size_test, y_size_pred, 1)
p = np.poly1d(z)
x_fit = np.linspace(min_val, max_val, 100)
ax.plot(x_fit, p(x_fit), 'r-', linewidth=1.5, alpha=0.8, label=f'Linear Fit (slope={z[0]:.3f})')

ax.set_xlabel('Actual Shred Size (mm)', fontsize=12)
ax.set_ylabel('Predicted Shred Size (mm)', fontsize=12)
ax.set_title(f'Actual vs Predicted Scatter (R²={r2_size:.4f})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_aspect('equal')
plt.colorbar(scatter, ax=ax, label='|Error| (mm)', shrink=0.8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("8-3. 산점도 시각화 완료")

### 8-4. 예측 오차 분포 히스토그램

오차가 0 근처에 집중되어 있을수록 모델 성능이 좋습니다.

In [ ]:
# 8-4. Error Distribution Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 파쇄 크기 오차
errors_size = y_size_test - y_size_pred
axes[0].hist(errors_size, bins=60, color='#3498db', alpha=0.8, edgecolor='white', density=True)
axes[0].axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2, label='Zero Error')
axes[0].axvline(x=np.mean(errors_size), color='#f39c12', linestyle='-', linewidth=2,
                label=f'Mean={np.mean(errors_size):.3f}mm')
axes[0].set_xlabel('Prediction Error (mm)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Shred Size Error Distribution (std={np.std(errors_size):.3f}mm)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# 균일도 오차
errors_unif = y_unif_test - y_unif_pred
axes[1].hist(errors_unif, bins=60, color='#2ecc71', alpha=0.8, edgecolor='white', density=True)
axes[1].axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2, label='Zero Error')
axes[1].axvline(x=np.mean(errors_unif), color='#f39c12', linestyle='-', linewidth=2,
                label=f'Mean={np.mean(errors_unif):.3f}%')
axes[1].set_xlabel('Prediction Error (%)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Uniformity Error Distribution (std={np.std(errors_unif):.3f}%)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()
print("8-4. 오차 분포 히스토그램 완료")

### 8-5. 균일도 예측 결과

균일도(%)의 실제값과 예측값을 시계열 및 산점도로 비교합니다.

In [ ]:
# 8-5. Uniformity Prediction Results
fig, axes = plt.subplots(2, 1, figsize=(16, 9))

# 시계열 비교
axes[0].plot(ts_plot, y_unif_test, alpha=0.6, linewidth=0.8, color='#2ecc71', label='Actual')
axes[0].plot(ts_plot, y_unif_pred, alpha=0.7, linewidth=0.8, color='#e67e22', label='Predicted (RF)')
axes[0].fill_between(ts_plot, y_unif_test, y_unif_pred, alpha=0.15, color='#f39c12')
axes[0].set_xlabel('Timestamp')
axes[0].set_ylabel('Uniformity (%)')
axes[0].set_title(f'Uniformity: Actual vs Predicted (R²={r2_unif:.4f}, MAE={mae_unif:.2f}%)',
                  fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].grid(True, alpha=0.3)

# 산점도
errors_u = np.abs(y_unif_test - y_unif_pred)
scatter = axes[1].scatter(y_unif_test, y_unif_pred, c=errors_u, cmap='RdYlGn_r',
                          alpha=0.4, s=10, edgecolors='none')
min_u = min(y_unif_test.min(), y_unif_pred.min())
max_u = max(y_unif_test.max(), y_unif_pred.max())
axes[1].plot([min_u, max_u], [min_u, max_u], 'k--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Uniformity (%)')
axes[1].set_ylabel('Predicted Uniformity (%)')
axes[1].set_title(f'Uniformity: Actual vs Predicted Scatter', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_aspect('equal')
plt.colorbar(scatter, ax=axes[1], label='|Error| (%)', shrink=0.8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("8-5. 균일도 예측 결과 시각화 완료")

### 8-6. 운전 조건별 파쇄 크기 분포

주간 정상 가동, 야간 저부하, 이벤트 발생 시 파쇄 크기가 어떻게 달라지는지 히스토그램으로 비교합니다.

In [ ]:
# 8-6. Size Distribution by Operating Condition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 데이터에서 운전 조건 분류
test_df = df.iloc[split_idx:].copy()
test_df['shred_size'] = y_size_test
test_df['hour'] = test_df['timestamp'].dt.hour
test_df['dow'] = test_df['timestamp'].dt.dayofweek

# 운전 조건 분류
conditions = []
for _, row in test_df.iterrows():
    if row['dow'] >= 5:
        conditions.append('Weekend (Off)')
    elif row['hour'] < 8 or row['hour'] >= 18:
        conditions.append('Night Shift')
    elif row['event_label'] != 'normal':
        conditions.append('Anomaly Event')
    else:
        conditions.append('Day Shift (Normal)')
test_df['condition'] = conditions

# 조건별 히스토그램
cond_colors = {
    'Day Shift (Normal)': '#2ecc71',
    'Night Shift': '#3498db',
    'Weekend (Off)': '#95a5a6',
    'Anomaly Event': '#e74c3c'
}
for cond, color in cond_colors.items():
    subset = test_df[test_df['condition'] == cond]['shred_size']
    if len(subset) > 0:
        axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=f'{cond} (n={len(subset):,})',
                     edgecolor='white', density=True)

axes[0].set_xlabel('Shred Size (mm)')
axes[0].set_ylabel('Density')
axes[0].set_title('Shred Size Distribution by Operating Condition', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# 실제 vs 예측 분포 비교
axes[1].hist(y_size_test, bins=50, alpha=0.6, color='#2563eb', label='Actual', edgecolor='white', density=True)
axes[1].hist(y_size_pred, bins=50, alpha=0.5, color='#dc2626', label='Predicted', edgecolor='white', density=True)
axes[1].axvline(x=np.mean(y_size_test), color='#2563eb', linestyle='--', linewidth=2,
                label=f'Actual Mean={np.mean(y_size_test):.1f}mm')
axes[1].axvline(x=np.mean(y_size_pred), color='#dc2626', linestyle='--', linewidth=2,
                label=f'Predicted Mean={np.mean(y_size_pred):.1f}mm')
axes[1].set_xlabel('Shred Size (mm)')
axes[1].set_ylabel('Density')
axes[1].set_title('Actual vs Predicted Size Distribution', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print("8-6. 운전 조건별 파쇄 크기 분포 시각화 완료")

## Step 9. 종합 요약

In [ ]:
# 피처 중요도 Top 3
importance_desc = importance.sort_values(ascending=False)
top3 = importance_desc.head(3)

print("=" * 65)
print("  Random Forest 파쇄 크기 예측 — 종합 요약")
print("=" * 65)

print(f"""
  [데이터 규모]
    전체 샘플: {len(df):,}개 (365일 x 10분 간격)
    학습: {len(X_train):,}개 / 테스트: {len(X_test):,}개
    피처: 12개 (4센서 그룹 x 3통계)

  [파쇄 크기 예측 성능]
    MAE  = {mae_size:.4f} mm
    RMSE = {rmse_size:.4f} mm
    R²   = {r2_size:.4f}

  [균일도 예측 성능]
    MAE  = {mae_unif:.4f} %
    RMSE = {rmse_unif:.4f} %
    R²   = {r2_unif:.4f}

  [피처 중요도 Top 3 (파쇄 크기)]
    1위: {top3.index[0]} = {top3.values[0]:.4f}
    2위: {top3.index[1]} = {top3.values[1]:.4f}
    3위: {top3.index[2]} = {top3.values[2]:.4f}

  [Random Forest 장점]
    - 비선형 관계 학습 가능 (센서 간 복합 상호작용)
    - Feature Importance로 핵심 센서 파악 가능
    - Bagging으로 과적합에 강건
    - 하이퍼파라미터 튜닝 없이도 양호한 성능

  [Random Forest 한계]
    - 학습 데이터 범위 밖의 외삽(Extrapolation) 불가
    - 200개 트리 → 모델 크기 및 추론 시간 증가
    - 시계열 의존성 직접 모델링 불가 (→ LSTM 등 보완 필요)

  [실무 적용 방안]
    - 실시간 파쇄 크기 모니터링: 10분마다 예측 → 품질 이상 조기 경보
    - 칼날 교체 시점 판단: 파쇄 크기 증가 추세 감지
    - RPM 최적화 연동: 모델 9 (PID Control)에 입력으로 활용
""")

print("=" * 65)
print("  노트북 실행 완료")
print("=" * 65)